In [1]:
from pathlib import Path
import pandas as pd

In [4]:
BASE = Path("/scratch/shelvia.wongso/cbo_results/ucbopt/cifar10_resnet20")

METRICS = ["acc", "acc@5", "nll", "ece","brier"]

rows = []

CURVS = ["0.0", "1e-6", "5e-6", "1e-5", "3e-5", "5e-5"]

nr = 0.2

rows = []

for curv in CURVS:
    pattern = f"lr_3.0_wd_2e-4_h0_0.5_curv_{curv}_tf_0.2_nr_{nr}" # ucbopt
    matched_dirs = sorted(BASE.glob(pattern))

    if not matched_dirs:
        print(f"[WARN] No folder found for lr={lr}")
        continue

    for curv_dir in matched_dirs:
        seed_best_rows = []

        for seed in range(3):
            seed_dir = curv_dir / f"seed={seed}"
            if not seed_dir.exists():
                print(f"[WARN] Missing {seed_dir}")
                continue

            run_dirs = sorted([d for d in seed_dir.iterdir() if d.is_dir()])
            if not run_dirs:
                print(f"[WARN] No run dirs in {seed_dir}")
                continue

            run_dir = run_dirs[-1]   # latest run
            test_csv = run_dir / "test.csv"
            if not test_csv.exists():
                print(f"[WARN] Missing {test_csv}")
                continue

            df = pd.read_csv(test_csv)

            best_idx = df["nll"].idxmin()
            best_row = df.loc[best_idx].copy()
            best_row["curv_dir"] = curv_dir.name
            best_row["seed"] = seed
            seed_best_rows.append(best_row)

        if not seed_best_rows:
            continue

        seed_best_df = pd.DataFrame(seed_best_rows)
        summary = {
            "curv": curv,
            "curv_dir": curv_dir.name,
            "num_seeds": len(seed_best_df),
        }

        for m in METRICS:
            if m in seed_best_df.columns:
                summary[f"{m}_mean"] = seed_best_df[m].mean()
                summary[f"{m}_std"] = seed_best_df[m].std(ddof=1)

        rows.append(summary)

summary_df = pd.DataFrame(rows).sort_values("nll_mean").reset_index(drop=True)

cols = [
    "curv", "num_seeds",
    "acc_mean", "acc_std",
    "acc@5_mean", "acc@5_std",
    "nll_mean", "nll_std",
    "ece_mean", "ece_std",
    "brier_mean", "brier_std",
]
cols = [c for c in cols if c in summary_df.columns]

print(summary_df[cols].to_string(index=False))

curv  num_seeds  acc_mean  acc_std  acc@5_mean  acc@5_std  nll_mean  nll_std  ece_mean  ece_std  brier_mean  brier_std
 0.0          3  0.714567 0.001012    0.968633   0.002281  0.922462 0.005122  0.105925 0.022461    0.409565   0.003642
5e-6          3  0.709233 0.006152    0.959133   0.007882  0.928111 0.012200  0.060029 0.027705    0.409687   0.003381
3e-5          3  0.709300 0.013157    0.960767   0.002926  0.928115 0.031844  0.081438 0.025090    0.409915   0.017433
1e-6          3  0.709300 0.002516    0.962833   0.002902  0.932975 0.010270  0.099411 0.007495    0.411634   0.003853
1e-5          3  0.707900 0.009440    0.954967   0.002230  0.940094 0.018163  0.087849 0.021260    0.411361   0.008795
5e-5          3  0.449233 0.020851    0.894433   0.020451  1.568650 0.043210  0.055395 0.026330    0.687052   0.012639


In [5]:
display_df = summary_df[cols].copy()

for col in display_df.columns:
    if col in ["lr", "lr_dir", "num_seeds"]:
        continue
    elif "acc" in col:
        display_df[col] = display_df[col].apply(lambda x: f"{x*100:.2f}" if pd.notnull(x) else "")
    else:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if pd.notnull(x) else "")

print(display_df.to_string(index=False))

ValueError: Unknown format code 'f' for object of type 'str'